In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Standalone 5-Class Soft Probability Weight Optimization (`models/optimize_soft_pipeline_thresholds.ipynb`)

This notebook optimizes the 5-class decision multiplier weights $\mathbf{w} = [w_1, w_2, w_3, w_4, w_5]$ for the **3-Tier Soft Probabilistic Joint Product Triage Pipeline**:

$$\hat{y}_i(\mathbf{w}) = \arg\max_{k \in \{1..5\}} \left( w_k \times P(\text{ESI } k) \right)$$

### Optimization Methodology
1. **Optimization Target**: Maximize **Macro Balanced Accuracy** of the combined 5-class system.
2. **Data Partitioning**:
   - **Validation Set**: Used strictly for Optuna weight tuning (200 trials). Must remain in its **natural, un-sampled class distribution**.
   - **1% Holdout Test Set**: Reserved for unbiased final evaluation.
3. **Resampling Strategy**:
   - **Sub-model Training**: Majority downsampling is applied during sub-model training (L1, L2, L3B) to produce informative, well-calibrated probabilities.
   - **Threshold Optimization**: **No resampling** is applied during weight tuning on validation data to preserve real-world population proportions.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Sub-Models in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}

fit_scaler <- function(df_train, cols) {
  means <- colMeans(df_train[, cols, drop = FALSE], na.rm = TRUE)
  sds   <- apply(df_train[, cols, drop = FALSE], 2, sd, na.rm = TRUE)
  sds[sds == 0] <- 1
  return(list(means = means, sds = sds, cols = cols))
}

apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}

pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")

df_master <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  hr_mean_to_last         = t_hr - pulse_last,
  spo2_mean_to_last       = t_o2 - spo2_last,
  rr_mean_to_last         = t_rr - resp_last,
  hr_range                = pulse_max - pulse_min,
  rr_range                = resp_max - resp_min,
  spo2_range              = spo2_max - spo2_min,
  sbp_range               = sbp_max - sbp_min,
  hr_last_to_min          = pulse_last - pulse_min,
  rr_last_to_min          = resp_last - resp_min,
  spo2_last_to_max        = spo2_last - spo2_max,
  hr_last_to_max          = pulse_last - pulse_max,
  sbp_last_to_max         = sbp_last - sbp_max,
  rr_last_to_max          = resp_last - resp_max
)

l1_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_hypertension", "is_tachypnea", "is_bradypnea", "is_tachycardia_total")
l2_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")
l3a_feat_names <- c("age", "gender", "cc_breathingdifficulty", "hr_mean_to_last", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "spo2_mean_to_last", "rr_mean_to_last")
l3b_feat_names <- c("age", "gender", "cc_breathingdifficulty", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "is_dyspnea_total", "hr_mean_to_last")

raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- stratified_partition(train_val_df$target_col, p = 1 - rel_val_size, seed = config$training$random_state)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

cont_cols <- c("age", "hr_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")

scaler <- fit_scaler(train_df, cont_cols)
val_scaled  <- apply_scaler(val_df, scaler)
test_scaled <- apply_scaler(test_df, scaler)

# Load Sub-Model Artifacts
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"

lgb_l1_obj  <- readRDS(file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
lgb_l2_obj  <- readRDS(file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
lgb_l3a_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi23_model.rds"))
lgb_l3b_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi45_model.rds"))

l1_model  <- if (is.list(lgb_l1_obj)  && "model" %in% names(lgb_l1_obj))  lgb_l1_obj$model  else lgb_l1_obj
l2_model  <- if (is.list(lgb_l2_obj)  && "model" %in% names(lgb_l2_obj))  lgb_l2_obj$model  else lgb_l2_obj
l3a_model <- if (is.list(lgb_l3a_obj) && "model" %in% names(lgb_l3a_obj)) lgb_l3a_obj$model else lgb_l3a_obj
l3b_model <- if (is.list(lgb_l3b_obj) && "model" %in% names(lgb_l3b_obj)) lgb_l3b_obj$model else lgb_l3b_obj

# Compute Validation Soft Probabilities
p1_val  <- predict(l1_model,  as.matrix(val_scaled[, l1_feat_names]))
p2_val  <- predict(l2_model,  as.matrix(val_scaled[, l2_feat_names]))
p3a_val <- predict(l3a_model, as.matrix(val_scaled[, l3a_feat_names]))
p3b_val <- predict(l3b_model, as.matrix(val_scaled[, l3b_feat_names]))

probs_val <- matrix(0, nrow = nrow(val_df), ncol = 5)
probs_val[, 1] <- p1_val
probs_val[, 2] <- (1 - p1_val) * p2_val * p3a_val
probs_val[, 3] <- (1 - p1_val) * p2_val * (1 - p3a_val)
probs_val[, 4] <- (1 - p1_val) * (1 - p2_val) * p3b_val
probs_val[, 5] <- (1 - p1_val) * (1 - p2_val) * (1 - p3b_val)
y_val_act <- as.numeric(as.character(val_df$target_col))

# Compute Test Soft Probabilities
p1_test  <- predict(l1_model,  as.matrix(test_scaled[, l1_feat_names]))
p2_test  <- predict(l2_model,  as.matrix(test_scaled[, l2_feat_names]))
p3a_test <- predict(l3a_model, as.matrix(test_scaled[, l3a_feat_names]))
p3b_test <- predict(l3b_model, as.matrix(test_scaled[, l3b_feat_names]))

probs_test <- matrix(0, nrow = nrow(test_df), ncol = 5)
probs_test[, 1] <- p1_test
probs_test[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_test[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_test[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_test[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
y_test_act <- as.numeric(as.character(test_df$target_col))

cat(sprintf("Validation Probabilities Matrix Prepared (%d x 5)\n", nrow(probs_val)))
cat(sprintf("Test Probabilities Matrix Prepared (%d x 5)\n", nrow(probs_test)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Optuna Weight Optimization on Validation Set
# ---------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import optuna
from rpy2.robjects import r

# Fetch Validation Probabilities & Ground Truth from R
P_val = np.array(r('probs_val'))
y_val = np.array(r('y_val_act'))

P_test = np.array(r('probs_test'))
y_test = np.array(r('y_test_act'))

def compute_macro_balanced_acc(y_true, y_pred):
    classes = [1, 2, 3, 4, 5]
    bal_accs = []
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        tn = np.sum((y_true != cls) & (y_pred != cls))
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((sens + spec) / 2.0)
    return np.mean(bal_accs)

def objective(trial):
    w1 = trial.suggest_float('w1', 0.1, 10.0, log=True)
    w2 = trial.suggest_float('w2', 0.1, 10.0, log=True)
    w3 = trial.suggest_float('w3', 0.1, 10.0, log=True)
    w4 = trial.suggest_float('w4', 0.1, 10.0, log=True)
    w5 = trial.suggest_float('w5', 0.1, 10.0, log=True)
    
    weights = np.array([w1, w2, w3, w4, w5])
    weighted_probs = P_val * weights
    preds = np.argmax(weighted_probs, axis=1) + 1
    return compute_macro_balanced_acc(y_val, preds)

# Run Optuna Study (200 Trials)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=200)

best_weights = study.best_params
best_val_score = study.best_value

print("=== Optuna Optimization Complete (200 Trials) ===")
print(f"  Best Validation Macro Balanced Accuracy: {best_val_score:.4f}")
print("  Optimal Class Multiplier Weights:")
for k, v in best_weights.items():
    print(f"    {k}: {v:.4f}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Evaluate Baseline vs Optimized Model on Holdout Test Set
# ---------------------------------------------------------
# Unweighted Baseline (w = [1, 1, 1, 1, 1])
base_preds_test = np.argmax(P_test, axis=1) + 1
base_bal_acc = compute_macro_balanced_acc(y_test, base_preds_test)

# Optimized Multiplier Weights
opt_weights = np.array([best_weights['w1'], best_weights['w2'], best_weights['w3'], best_weights['w4'], best_weights['w5']])
opt_preds_test = np.argmax(P_test * opt_weights, axis=1) + 1
opt_bal_acc = compute_macro_balanced_acc(y_test, opt_preds_test)

def get_detailed_metrics(y_true, y_pred):
    classes = [1, 2, 3, 4, 5]
    recalls, specs, bal_accs = [], [], []
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        tn = np.sum((y_true != cls) & (y_pred != cls))
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        recalls.append(sens)
        specs.append(spec)
        bal_accs.append((sens + spec) / 2.0)
        
    return {
        'Macro_Recall': np.mean(recalls),
        'Macro_Specificity': np.mean(specs),
        'Macro_Balanced_Accuracy': np.mean(bal_accs)
    }

base_metrics = get_detailed_metrics(y_test, base_preds_test)
opt_metrics  = get_detailed_metrics(y_test, opt_preds_test)

summary_df = pd.DataFrame([
    {'Pipeline': 'Baseline_Unweighted_Soft_Model', **base_metrics},
    {'Pipeline': 'Optuna_Weighted_Soft_Model', **opt_metrics}
])

print("============================================================")
print("   HOLDOUT TEST SET PERFORMANCE COMPARISON")
print("============================================================")
print(summary_df.to_string(index=False))
print("============================================================\n")

# Save Optimized Weights JSON
opt_weights_dict = {
    'weights': [float(best_weights[f'w{i}']) for i in range(1, 6)],
    'val_macro_balanced_accuracy': float(best_val_score),
    'test_macro_balanced_accuracy': float(opt_metrics['Macro_Balanced_Accuracy'])
}

deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(deploy_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

with open(os.path.join(deploy_dir, 'soft_pipeline_optimized_weights.json'), 'w') as f:
    json.dump(opt_weights_dict, f, indent=2)

summary_df.to_csv(os.path.join(reports_dir, 'optimized_soft_pipeline_report.csv'), index=False)
print("Artifacts Saved:")
print(f"  - {os.path.join(deploy_dir, 'soft_pipeline_optimized_weights.json')}")
print(f"  - {os.path.join(reports_dir, 'optimized_soft_pipeline_report.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Plot Comparison Bar Chart
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_melted = pd.melt(summary_df, id_vars=['Pipeline'], var_name='Metric', value_name='Score')

plt.figure(figsize=(9, 5.5))
ax = sns.barplot(data=df_melted, x='Metric', y='Score', hue='Pipeline', palette=['#1f77b4', '#2ca02c'])

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.4f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=10, xytext=(0, 3),
                    textcoords='offset points')

plt.title('Soft Probabilistic Triage Model: Baseline vs Optuna Weight Optimization', fontsize=14, fontweight='bold', pad=15)
plt.ylim(0, 1.1)
plt.ylabel('Score', fontsize=12)
plt.xlabel('Macro Metric', fontsize=12)
plt.legend(title='Pipeline', loc='upper left')

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'optimized_soft_pipeline_comparison.png'), dpi=300)
plt.show()
print(f"Plot saved to {os.path.join(plots_dir, 'optimized_soft_pipeline_comparison.png')}")